# 02 — Configure and check the live Inference.sh connection

**Purpose:** send one synthetic state and one NOUL question to one selected JEV model through the implemented Inference.sh backend.

| Item | Description |
|---|---|
| Connection configuration | `configs/connections/inference_sh.yaml` |
| Selected model | Selected by `model_config_file`; switch between the JEV-9B and JEV-27B YAML files |
| Inputs | The notebook-only synthetic study and question below |
| Outputs | HTTP status, credential-redacted JSON, validated probabilities, and automatically saved live-call artifacts |
| Mode | Live; executing the request cell can incur API charges |

The separate `configs/notebooks.yaml` demo setting belongs to the general research/batch walkthrough. This notebook uses its own connection settings. Inference.sh batch execution remains blocked. Live calls automatically save under ignored `results/connections/`; mock calls do not write by default.

## 1. Load settings and inspect model selection

Use the project's `.venv` kernel. Restart the kernel after source-code updates, then run from the top. Initialization loads the local `.env` without overwriting existing environment variables. The request uses `INFERENCE_API_KEY`; its value is never displayed.

The connection YAML now selects `execution.mode: live`. Model app IDs come from model YAML, not the draft's mutable `jev-latest` alias.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select the project's .venv kernel. Run uv sync --locked --group notebooks "
        "from the repository terminal if dependencies are missing."
    ) from None

from jev_eval.connections import connection_readiness, load_connection

project = initialize_notebook()
connection = load_connection(project.root)

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "model": connection.model.name,
            "description": connection.model.display_name,
            "app_id": connection.model.hosted.app_id,
            "connection_mode": connection.settings.execution.mode,
        }
    ]
)

## 2. Inspect hosted prerequisites without sending a request

The Inference.sh backend is implemented. The table reports credential availability without showing its value. No request is sent by this readiness check. Set `INFERENCE_API_KEY` in the ignored `.env` if it is missing, then restart the kernel.

Switch models through `model_config_file` in the connection YAML: `configs/models/jev-9b.yaml` selects `autotrust/jev-9b`; `configs/models/jev-27b.yaml` selects `autotrust/jev-27b-v1-1`.

In [ ]:
connection_readiness(connection)

## 3. Define and validate a synthetic screening example

This invented study is for API connectivity only. It is not part of the evaluation dataset and does not replace the evaluation screening protocol. The configured file-based task remains available as `connection`; the example is a separate derived connection.

One NOUL asks for the probability that the answer is yes. The hybrid draft is not selected or automatically aggregated here.

In [ ]:
from jev_eval.connections import with_synthetic_example

state = {
    "title": "Effects of an antidepressant on depressive-like behaviour in mice",
    "abstract": (
        "Adult mice were randomly assigned to receive an "
        "antidepressant or vehicle control for 14 days. "
        "Depressive-like behaviour was measured using "
        "the forced swim test."
    ),
}
question = {
    "id": "screening_eligibility",
    "instructions": (
        "Does this study report an in vivo experiment "
        "in mice or rats evaluating an antidepressant "
        "intervention with a behavioural outcome?"
    ),
}


example_connection = with_synthetic_example(connection, state=state, question=question)

In [ ]:
pd.DataFrame(
    [
        {"question": name, "type": value.type, "instructions": value.instructions}
        for name, value in example_connection.task.questions.items()
    ]
)

## 4. Preview one request without sending it

The request contains the selected `app`, the synthetic `state`, `nouls: [question]`, and `wait: true`. Authentication is sent separately by the backend, so no credential appears in this preview. This is one state, one question, and one model, with no retries or batching.

In [ ]:
from jev_eval.connections import preview_connection

preview_connection(example_connection)

## 5. Execute one live request — potentially billable

**Paid API warning:** executing the next cell with the current live settings sends one request to `https://api.inference.sh/apps/run` and may incur charges. `ALLOW_PAID_CALLS` is enabled below to match your requested switch to live execution. Set it to false to preview without sending, or change connection YAML to `mode: mock` for an offline response.

Each rerun can create another chargeable task. A timeout may mean the provider already accepted it; do not automatically repeat the POST. There are no automatic retries or batch inference.

**File-writing warning:** live calls automatically save the request, exact response bytes, validated answers, and provenance in a unique directory under ignored `results/connections/`. `output.allow_writes: true` is required in connection YAML. The helper checks storage before sending; the saved directory is printed below. Mock calls do not write by default.

The cell displays HTTP status and credential-redacted JSON. Authentication failures, HTTP errors, and unexpected response formats have informative messages. The exact raw bytes remain in `result.exchange` and the saved `response.body`; never print credentials or request headers. If saving fails after the call, keep the kernel open and use Section 6 to save the retained result without another API request.

In [ ]:
import json

from jev_eval.config import PrerequisiteError
from jev_eval.connections import (
    ConnectionPersistenceError,
    connection_response_view,
    run_connection,
    summarize_connection,
)

ALLOW_PAID_CALLS = True
result = None
if example_connection.settings.execution.mode == "mock" or ALLOW_PAID_CALLS:
    try:
        result = run_connection(example_connection, allow_paid_calls=ALLOW_PAID_CALLS)
    except ConnectionPersistenceError as error:
        result = error.result
        print("SAVE FAILED: the API result is retained in memory.")
        print("Do not rerun this request cell. Fix storage and use Section 6 to save only.")
    except PrerequisiteError as error:
        print(f"Connection not started: {error}")
    if result is not None:
        view = connection_response_view(result)
        print("HTTP status:", view["http_status"])
        print(view["message"])
        if view["is_json"]:
            print(json.dumps(view["raw_json"], indent=2, ensure_ascii=False))
        display(summarize_connection(result))
        if result.directory is not None:
            print("Saved artifacts:", result.directory)
else:
    print("Request skipped: ALLOW_PAID_CALLS is disabled.")

## 6. Recover an unsaved result without resending

Successful live calls are saved automatically; this step is only needed if Section 5 reports a storage failure. The retained `result` contains the original request, response, and provenance. Fix the local storage problem first and keep this kernel open.

**File-writing warning:** set `RECOVER_UNSAVED_RESULT = True` to write the retained artifacts. This cell makes no API request and creates no new model charge. It leaves an already saved result alone. If a failed write left a partial directory, use the new directory printed by the successful save as the complete record.

In [ ]:
from jev_eval.connections import save_connection

RECOVER_UNSAVED_RESULT = False
if result is None:
    print("No result is available in this kernel. Do not resend to recover an earlier response.")
elif result.directory is not None:
    print("Already saved:", result.directory)
elif not RECOVER_UNSAVED_RESULT:
    print("Save-only recovery is disabled. Enable it after fixing storage if needed.")
else:
    try:
        saved = save_connection(result, example_connection, allow_writes=True)
        print("Saved artifacts without another API request:", saved)
    except (OSError, PrerequisiteError):
        print(
            "Saving still failed; result remains in memory. Check storage and YAML write permission."
        )

## 7. Future local inference and batch execution

Local inference remains separate and unavailable; no GPU packages or model weights are installed by this workflow. The shared interface supports a future local adapter.

Inference.sh bulk execution is not enabled. Notebook 03 is a separate demo/generic-runner workflow and does not batch this provider. Responses here are connection checks rather than evaluation results. Clear all notebook outputs before saving or staging for Git; the existing commit hook rejects saved outputs.